# NB 22 — Day-0 Survey Calibration (Claude Sonnet, debiased)

Targets the **survey path** of the production stack: Claude Sonnet under the
two-step debiasing protocol, no extended thinking, temperature 0.5.

For each of the 30 simulation agents (same YouGov sample as the main runs,
`random_state=43`), elicit a Day-0 opinion on each of the 6 climate policies
and compare against the agent's real YouGov response.

180 calls total. Captures: letter, numeric (-3..+3), ground truth, signed
error, absolute error, exact match, ordinal score (|err| ≤ 1).

Also runs an internal permutation null: 100 random shuffles of the
agent → ground-truth mapping on the same LLM responses, to check whether
per-policy accuracy exceeds chance pairing.

Output: `data/output/calibration/<timestamp>/`


## 1. Imports


In [1]:
import os, sys, random, logging, json, time
from datetime import datetime
from pathlib import Path

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '../src')))
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')
logging.getLogger("httpx").setLevel(logging.WARNING)
logging.getLogger("httpcore").setLevel(logging.WARNING)
logging.getLogger("anthropic").setLevel(logging.WARNING)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

from cag.io.survey import load
from cag.io.llm import load_api_key
from cag.abm.agent import SurveyedCitizen
from cag.abm.environment import SurveyedNation
from cag.abm.attributes.opinion import ClimatePolicyID, SURVEY_QUESTIONS

from gabm.abm.attributes.gender import GenderMap, GenderID
from gabm.abm.attributes.politics import PoliticsID
from gabm.abm.democracy.election import ElectionID
from cag.abm.attributes.region import UKRegionMap, RegionID
from cag.abm.attributes.education import SurveyEducationMap, EducationID
from cag.abm.attributes.ethnicity import SurveyEthnicityMap, EthnicityID
from cag.abm.attributes.income import SurveyIncomeMap, IncomeID
from cag.abm.attributes.politics import SurveyPoliticsMap
from cag.abm.attributes.family import SurveyFamilyMap, FamilyID
from cag.abm.democracy.elections.ukge2019 import UKGE2019VoteMap, UKGE2019VoteID
from cag.abm.democracy.elections.brexit import BrexitVoteMap, BrexitVoteID
from cag.abm.attributes.narratives import (
    SelftranscMap, SelfenhMap, OpennessMap, ConformTradMap, SDOMap, EDOMap, RWAMap,
    rescale_1_6, rescale_1_7,
)

print("Imports OK")


Imports OK


## 2. Configuration

Mirrors the production survey path: Claude Sonnet, two-step debias, no extended thinking, T=0.5. Same agent sample as the main runs (`random_state=43`).


In [2]:
# --- Config: mirrors the production survey path of the main runs ---
N_AGENTS    = 30
RANDOM_SEED = 43               # same as Run 5 onward
MODEL       = "claude-sonnet-4-6"
PROVIDER    = "anthropic"
TEMPERATURE = 0.5
THINKING    = False
DEBIAS      = True
N_PERMS     = 100

POLICIES = list(SURVEY_QUESTIONS.keys())   # 6 ClimatePolicyID values
api_key  = load_api_key(PROVIDER)

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_DIR = Path(f"../data/output/calibration/{ts}")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"N_AGENTS={N_AGENTS}  POLICIES={len(POLICIES)}  total calls={N_AGENTS*len(POLICIES)}")
print(f"model={MODEL} provider={PROVIDER} debias={DEBIAS} thinking={THINKING} temp={TEMPERATURE}")
print(f"output → {OUT_DIR}")


N_AGENTS=30  POLICIES=6  total calls=180
model=claude-sonnet-4-6 provider=anthropic debias=True thinking=False temp=0.5
output → ../data/output/calibration/20260425_203841


## 3. Build the citizen agents

Same `SurveyedNation` construction as the main simulation runs. Each row of the YouGov sample becomes one `SurveyedCitizen` with the full demographic + values persona attached.


In [3]:
# --- Build SurveyedNation and load 30 agents (same sample as Run 5+) ---
random.seed(RANDOM_SEED)
year = 2026
UKGE2019_ELECTION_ID = ElectionID(0)
BREXIT_REFERENDUM_ID = ElectionID(1)

sn = SurveyedNation(
    year=year, place="UK",
    gender_map=GenderMap(),
    region_map=UKRegionMap(),
    education_map=SurveyEducationMap(),
    ethnicity_map=SurveyEthnicityMap(),
    income_map=SurveyIncomeMap(),
    politics_map=SurveyPoliticsMap(),
    family_map=SurveyFamilyMap(),
    ukge2019_vote_map=UKGE2019VoteMap(UKGE2019_ELECTION_ID),
    brexit_vote_map=BrexitVoteMap(BREXIT_REFERENDUM_ID),
    selftransc_map=SelftranscMap, selfenh_map=SelfenhMap,
    openness_map=OpennessMap, conformtrad_map=ConformTradMap,
    sdo_map=SDOMap, edo_map=EDOMap, rwa_map=RWAMap,
)

data = load("../data/yougov_survey_data/YouGovProcessedData.csv")
data = data.sample(n=N_AGENTS, random_state=RANDOM_SEED)

for i in range(len(data)):
    row = data.iloc[i]
    sc = SurveyedCitizen(
        agent_id=row.get('ID', None), environment=sn,
        year_of_birth=year - int(row.get('age', 0)),
        gender_id=GenderID.MALE if int(row.get('male_dummy', 0)) == 1 else GenderID.FEMALE,
        region_id=RegionID(int(row.get('tprofile_GOR', 0))),
        education_id=EducationID(int(row.get('profile_education_level', 0))),
        income_id=IncomeID(int(row.get('tprofile_gross_household', 0))),
        ethnicity_id=EthnicityID(int(row.get('ethnicity_R', 0))),
        family_id=FamilyID.PARENT if int(row.get('parent_dummy', 0)) == 1 else FamilyID.NOT_PARENT,
        ukge2019_vote_id=UKGE2019VoteID(int(row.get('Vote2019R', 0))),
        brexit_vote_id=BrexitVoteID(int(row.get('pastvote_EURef', 0))),
        politics_id=PoliticsID(int(row.get('Political_Left_Right', 0))),
        selftransc_id=rescale_1_6(int(row.get('Selftransc_Val', 0))),
        selfenh_id=rescale_1_6(int(row.get('Selfenh_Values', 0))),
        openness_id=rescale_1_6(int(row.get('Openness', 0))),
        conformtrad_id=rescale_1_6(int(row.get('ConformTrad', 0))),
        sdo_id=rescale_1_7(int(row.get('SDO', 0))),
        edo_id=rescale_1_7(int(row.get('EDO', 0))),
        rwa_id=rescale_1_6(int(row.get('RWA', 0))),
        original_survey_data=data.iloc[i],
    )
    sn.agents_active[sc.id] = sc

agents = list(sn.agents_active.values())
print(f"Loaded {len(agents)} agents")


INFO Columns with NaN counts (before filtering):
tprofile_gross_household    398
Political_Left_Right          6
dtype: int64
INFO Column with most NaNs: tprofile_gross_household (398 NaNs)
INFO 1483 rows after filtering.


Loaded 30 agents


## 4. Calibration loop (180 LLM calls)

For each (agent, policy) pair: clear the agent's `opinion_history`, administer the Day-0 debiased survey via Claude Sonnet, capture the letter response and its numeric mapping, and compare against the YouGov ground truth.

Each row records signed error, absolute error, exact match (numeric == GT) and ordinal match (|err| ≤ 1).


In [4]:
# --- Main calibration loop: 30 agents x 6 policies = 180 calls ---
records = []
t0 = time.time()
for i, agent in enumerate(agents):
    for policy_id in POLICIES:
        # Reset opinion_history for clean Day-0 elicitation
        agent.opinion_history = {}

        gt = agent.get_real_survey_response(policy_id)
        try:
            letter, numeric = agent.administer_survey(
                policy_id, day=0,
                model=MODEL, provider=PROVIDER, api_key=api_key,
                temperature=TEMPERATURE, thinking=THINKING, debias=DEBIAS,
            )
        except Exception as e:
            logging.warning(f"agent={agent.id} policy={policy_id} error: {e}")
            letter, numeric = None, None

        err = None if numeric is None else (numeric - gt)
        records.append({
            "agent_id":   agent.id,
            "policy_id":  str(policy_id),
            "letter":     letter,
            "llm":        numeric,
            "gt":         gt,
            "err":        err,
            "abs_err":    None if err is None else abs(err),
            "exact":      None if numeric is None else int(numeric == gt),
            "ordinal":    None if err is None else int(abs(err) <= 1),
        })
    if (i + 1) % 5 == 0:
        elapsed = time.time() - t0
        print(f"  agent {i+1}/{N_AGENTS}  elapsed={elapsed:.0f}s")

df = pd.DataFrame(records)
df.to_csv(OUT_DIR / "calibration_raw.csv", index=False)
print(f"\nDone. {len(df)} rows. Saved to {OUT_DIR/'calibration_raw.csv'}")
print(f"Failures: {df['llm'].isna().sum()}")
df.head()


  agent 5/30  elapsed=185s
  agent 10/30  elapsed=387s
  agent 15/30  elapsed=597s
  agent 20/30  elapsed=775s
  agent 25/30  elapsed=959s
  agent 30/30  elapsed=1154s

Done. 180 rows. Saved to ../data/output/calibration/20260425_203841/calibration_raw.csv
Failures: 0


,agent_id,policy_id,letter,llm,gt,err,abs_err,exact,ordinal
0,1923.0,ClimatePolicyID(1),F,2,1,1,1,0,1
1,1923.0,ClimatePolicyID(2),G,3,1,2,2,0,0
2,1923.0,ClimatePolicyID(3),F,2,1,1,1,0,1
3,1923.0,ClimatePolicyID(4),F,2,1,1,1,0,1
4,1923.0,ClimatePolicyID(5),F,2,0,2,2,0,0


## 5. Per-policy aggregates

For each of the 6 climate policies: exact-match rate, ordinal accuracy (|err| ≤ 1), MAE, signed bias (`llm − gt`, positive = LLM more pro-climate than reality), Spearman rank correlation, and the mean LLM vs mean GT response.

Read `bias` first: this is the persistent direction of error. `ordinal` and `mae` say how tight the predictions are around the truth.


In [5]:
# --- Per-policy aggregates ---
def _agg(g):
    return pd.Series({
        "n":         len(g),
        "exact":     g["exact"].mean(),
        "ordinal":   g["ordinal"].mean(),
        "mae":       g["abs_err"].mean(),
        "bias":      g["err"].mean(),
        "spearman":  spearmanr(g["llm"], g["gt"]).correlation if g["llm"].nunique() > 1 else np.nan,
        "llm_mean":  g["llm"].mean(),
        "gt_mean":   g["gt"].mean(),
    })

per_policy = df.dropna(subset=["llm"]).groupby("policy_id", sort=False).apply(_agg, include_groups=False).round(3)
per_policy.to_csv(OUT_DIR / "per_policy.csv")
per_policy


,n,exact,ordinal,mae,bias,spearman,llm_mean,gt_mean
policy_id,,,,,,,,
ClimatePolicyID(1),30.0,0.400,0.833,0.867,0.467,0.251,2.333,1.867
ClimatePolicyID(2),30.0,0.433,0.800,0.933,0.133,0.613,1.200,1.067
ClimatePolicyID(3),30.0,0.233,0.700,1.267,0.067,0.628,0.500,0.433
ClimatePolicyID(4),30.0,0.233,0.767,1.033,0.300,0.444,1.833,1.533
ClimatePolicyID(5),30.0,0.233,0.567,1.400,0.867,0.480,1.633,0.767
ClimatePolicyID(6),30.0,0.200,0.567,1.667,0.600,0.464,0.700,0.100


## 6. Per-agent aggregates and overall headline numbers

`per_agent` averages the four error metrics across the 6 policies for each agent (useful for spotting agents the LLM consistently mis-renders). `overall` is the pooled headline number across all 180 (agent, policy) pairs.


In [6]:
# --- Per-agent aggregates and headline numbers ---
per_agent = df.dropna(subset=["llm"]).groupby("agent_id").agg(
    n=("llm", "size"),
    exact=("exact", "mean"),
    ordinal=("ordinal", "mean"),
    mae=("abs_err", "mean"),
    bias=("err", "mean"),
).round(3)
per_agent.to_csv(OUT_DIR / "per_agent.csv")

valid = df.dropna(subset=["llm"])
overall = {
    "n":        int(len(valid)),
    "exact":    float(valid["exact"].mean()),
    "ordinal":  float(valid["ordinal"].mean()),
    "mae":      float(valid["abs_err"].mean()),
    "bias":     float(valid["err"].mean()),
    "spearman": float(spearmanr(valid["llm"], valid["gt"]).correlation),
}
print("OVERALL")
for k, v in overall.items():
    print(f"  {k:9s} {v:.3f}" if isinstance(v, float) else f"  {k:9s} {v}")


OVERALL
  n         180
  exact     0.289
  ordinal   0.706
  mae       1.194
  bias      0.406
  spearman  0.550


## 7. Permutation null

Per policy, hold the 30 LLM responses fixed and shuffle the agent → ground-truth pairing 100 times. `p_ord` is the fraction of shuffles whose ordinal accuracy meets or beats the real pairing; `p_mae` is the fraction whose MAE is at or below the real pairing.

`p < 0.05` ⇒ the real pairing genuinely beats chance. Note: on policies where GT has very compressed variance (e.g. Renewable energy, where most respondents are pro), random pairing already scores well, so the test has low power there.


In [7]:
# --- Permutation null: shuffle agent → ground-truth mapping per policy ---
rng = np.random.default_rng(RANDOM_SEED)
null_rows = []
for pid, g in valid.groupby("policy_id", sort=False):
    llm = g["llm"].to_numpy()
    gt  = g["gt"].to_numpy()
    real_ord = np.mean(np.abs(llm - gt) <= 1)
    real_mae = np.mean(np.abs(llm - gt))
    perms_ord, perms_mae = [], []
    for _ in range(N_PERMS):
        gt_shuf = rng.permutation(gt)
        perms_ord.append(np.mean(np.abs(llm - gt_shuf) <= 1))
        perms_mae.append(np.mean(np.abs(llm - gt_shuf)))
    perms_ord = np.array(perms_ord)
    perms_mae = np.array(perms_mae)
    null_rows.append({
        "policy_id":  pid,
        "real_ord":   round(real_ord, 3),
        "null_ord_mean": round(perms_ord.mean(), 3),
        "p_ord":      round((perms_ord >= real_ord).mean(), 3),
        "real_mae":   round(real_mae, 3),
        "null_mae_mean": round(perms_mae.mean(), 3),
        "p_mae":      round((perms_mae <= real_mae).mean(), 3),
    })

null_df = pd.DataFrame(null_rows)
null_df.to_csv(OUT_DIR / "permutation_null.csv", index=False)
null_df


,policy_id,real_ord,null_ord_mean,p_ord,real_mae,null_mae_mean,p_mae
0,ClimatePolicyID(1),0.833,0.745,0.04,0.867,1.045,0.07
1,ClimatePolicyID(2),0.800,0.443,0.00,0.933,2.133,0.00
2,ClimatePolicyID(3),0.700,0.423,0.00,1.267,2.244,0.00
3,ClimatePolicyID(4),0.767,0.600,0.00,1.033,1.496,0.00
4,ClimatePolicyID(5),0.567,0.481,0.13,1.400,1.885,0.01
5,ClimatePolicyID(6),0.567,0.400,0.02,1.667,2.178,0.04


## 8. Persist run metadata


In [8]:
# --- Save config and summary ---
summary = {
    "timestamp":    ts,
    "n_agents":     N_AGENTS,
    "policies":     [str(p) for p in POLICIES],
    "random_seed":  RANDOM_SEED,
    "model":        MODEL,
    "provider":     PROVIDER,
    "temperature":  TEMPERATURE,
    "thinking":     THINKING,
    "debias":       DEBIAS,
    "n_perms":      N_PERMS,
    "overall":      overall,
}
with open(OUT_DIR / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"All artefacts saved to {OUT_DIR}/")
for p in sorted(OUT_DIR.iterdir()):
    print(f"  {p.name}")


All artefacts saved to ../data/output/calibration/20260425_203841/
  calibration_raw.csv
  per_agent.csv
  per_policy.csv
  permutation_null.csv
  summary.json


## 9. Findings (run `20260425_203841`)

### Headline (180 (agent, policy) pairs)

| metric | value |
|---|---|
| Exact match | 28.9% |
| Ordinal accuracy (\|err\| ≤ 1) | 70.6% |
| MAE | 1.19 |
| Signed bias (`llm − gt`) | **+0.41** |
| Spearman ρ (LLM vs GT) | 0.55 |

### Per policy

| Policy | exact | ordinal | MAE | bias | ρ |
|---|---:|---:|---:|---:|---:|
| 1 Renewable energy | 0.40 | 0.83 | 0.87 | +0.47 | 0.25 |
| 2 Ban fossil fuels | 0.43 | 0.80 | 0.93 | +0.13 | 0.61 |
| 3 Ban petrol cars | 0.23 | 0.70 | 1.27 | +0.07 | 0.63 |
| 4 Green housing | 0.23 | 0.77 | 1.03 | +0.30 | 0.44 |
| 5 Carbon tax | 0.23 | 0.57 | 1.40 | **+0.87** | 0.48 |
| 6 Climate compensation | 0.20 | 0.57 | 1.67 | **+0.60** | 0.46 |

### Permutation null

| Policy | real ord | null ord mean | p_ord | p_mae |
|---|---:|---:|---:|---:|
| 1 Renewable energy | 0.83 | 0.75 | 0.04 | 0.07 |
| 2 Ban fossil fuels | 0.80 | 0.44 | **0.00** | **0.00** |
| 3 Ban petrol cars | 0.70 | 0.42 | **0.00** | **0.00** |
| 4 Green housing | 0.77 | 0.60 | **0.00** | **0.00** |
| 5 Carbon tax | 0.57 | 0.48 | 0.13 | 0.01 |
| 6 Climate compensation | 0.57 | 0.40 | 0.02 | 0.04 |

### Interpretation

1. **Rank correlation is moderate** (ρ = 0.55 overall, 0.61–0.63 on the polarised policies 2 and 3): Sonnet under debias does pick up real persona-level variation, not just a marginal distribution.
2. **A residual pro-climate bias remains** (+0.41). It is not uniform across policies — it concentrates on the two policies where the public is most lukewarm: **Carbon tax (+0.87)** and **Climate compensation (+0.60)**. The two concrete behavioural restrictions (Ban petrol cars +0.07, Ban fossil fuels +0.13) are essentially unbiased.
3. **Implication for the simulation**: any "support" trajectory on Carbon tax or Climate compensation in the main runs sits on top of a Day-0 anchor that is already shifted up by roughly two-thirds of a scale point. Movement on those policies should be read as movement *away from* a biased anchor, not as movement away from ground truth.
4. **Permutation null**: the real pairing beats chance at p < 0.05 on 4 of 6 policies (2, 3, 4, 6). Renewable energy is borderline (p = 0.04 on ordinal, 0.07 on MAE) and Carbon tax fails on ordinal (p = 0.13) — both are policies where the LLM's marginal distribution already overlaps the GT marginal, so random pairing scores well by accident. The MAE test passes on Carbon tax (p = 0.01), so the failure on `p_ord` is a power issue, not evidence against persona-conditioning.
5. **Weakness of this notebook**: only 30 agents per policy is a small sample for permutation power, and the per-agent data are tied to the same agents that drive Probe 1/2. NB 23 addresses both with 100 fresh personas on the two most informative policies (Renewable energy as the hard case, Ban petrol cars as the easy case).
